In [19]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.model_selection import KFold
    
def target_encode(df, target, col, n_splits=5, seed=42):
    """
    Target Encoding с K-Fold для признака col
    """
    df_encoded = df.copy()
    df_encoded[col + '_target'] = np.nan
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    
    for train_idx, val_idx in kf.split(df_encoded):
        train, val = df_encoded.iloc[train_idx], df_encoded.iloc[val_idx]
        means = train.groupby(col)[target].mean()
        df_encoded.loc[val_idx, col + '_target'] = val[col].map(means)
    
    return df_encoded

def frequency_encode(df, col):
    """Frequency Encoding"""
    freq = df[col].value_counts(normalize=True)
    return df[col].map(freq)

def glm_binomial_target(df, target):
    print(df.head())
    df_clean = df.copy()
    
    # Preprocessing and feature selection
    df_clean = df_clean.dropna(subset=[target])
    feature_cols = [c for c in df_clean.columns if c not in [target]]
    
    numeric_cols = [c for c in feature_cols if df_clean[c].dtype in ['int64', 'float64']]
    categorical_cols = [c for c in feature_cols if c not in numeric_cols]
    
    for col in numeric_cols:
        df_clean[col] = df_clean[col].fillna(df_clean[col].mean())
    for col in categorical_cols:
        df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])
    
    df_clean[target], _ = pd.factorize(df_clean[target])
    
    encoded_features = []
    for col in categorical_cols:
        df_clean = target_encode(df_clean, target, col)
        encoded_features.append(col + '_target')
    
    X = pd.concat([df_clean[numeric_cols], df_clean[encoded_features]], axis=1).astype(float)
    y = df_clean[target].astype(float)
                                
    X = sm.add_constant(X)

    model = sm.GLM(y, X, family=sm.families.Binomial(link=sm.families.links.Logit()))
    result = model.fit()
    
    # Проверки
    print(*X)
    nan_check = X.isna().sum().sum() == 0
    binary_check = all(
        set(X[col].unique()).issubset({0, 1}) 
        for col in X.columns if len(X[col].unique()) <= 2
    )
    print(f"NaN check passed: {nan_check}")
    print(f"Binary check passed: {binary_check}")
    
    # Interpretation
    summary_df = pd.DataFrame({
        'coef': result.params,
        'odds ratio': np.exp(result.params),
        'p-value': result.pvalues
    })
    summary_df['Interpretation'] = summary_df.apply(
    lambda row: (
        f"{'increases' if row['coef'] > 0 else 'decreases'} chances"
        + (f" by {row['odds ratio']:.2f}" if row['odds ratio'] > 0 else "")
        + (' (not significant)' if row['p-value'] > 0.05 else '')), axis=1)
    
    
    return result, summary_df

In [20]:
example = pd.read_csv('./smoke.csv')

result, summary_df =  glm_binomial_target(example, 'smoker')
print(result.summary())
print()
print('=====================Interpretation=======================================================')
print(summary_df)
print()
print()

  smoker  ban   age     education afam hispanic  gender
0    yes  yes  41.0            hs   no       no  female
1    yes  NaN  44.0  some college   no       no  female
2     no   no   NaN  some college   no       no  female
3    yes   no  29.0           NaN   no       no  female
4     no  yes  28.0  some college  NaN       no  female
const age ban_target education_target afam_target hispanic_target gender_target
NaN check passed: True
Binary check passed: True
                 Generalized Linear Model Regression Results                  
Dep. Variable:                 smoker   No. Observations:                10000
Model:                            GLM   Df Residuals:                     9993
Model Family:                Binomial   Df Model:                            6
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -5266.2
Date:                Wed, 17 Dec 2025   Deviance:            